# Data Exploration & Cleaning

## 1. Load the Data

I began by importing the Telco Customer Churn dataset from Kaggle using kagglehub and loading it into a pandas DataFrame for analysis.

In [2]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("blastchar/telco-customer-churn")

df = pd.read_csv("/Users/arkyaghosh/churn_project_data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

## 2. Initial Exploration

To understand the dataset structure and identify potential data quality issues, I performed an initial exploratory analysis.

In [24]:
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nColumn names and types:")
print(df.dtypes)
print("\nBasic info:")
print(df.info())
print("\n Statistic for numeric columns:")
print(df.describe())

Dataset shape: (7043, 21)

First few rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Strea

One thing that I noticed was that the TotalCharges column was string, despite the values in it being all floats.

## 3. Data Cleaning

**Problem: TotalCharges Data Type Issue**

When attempting to convert TotalCharges to float, I encountered a ValueError:


In [25]:
try:
    df["TotalCharges"] = df["TotalCharges"].astype(float)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: could not convert string to float: ' '


The error indicated that some values contained spaces or were empty, preventing conversion.

**Solution: Identify and Remove Problematic Rows**

I stripped whitespace and identified rows with missing TotalCharges values:

In [26]:
df["TotalCharges"] = df["TotalCharges"].str.strip()
tempSet = pd.to_numeric(df["TotalCharges"], errors="coerce")
mv = tempSet.isnull()

print(f"Rows with missing TotalCharges: {mv.sum()}")
print(df[mv])

Rows with missing TotalCharges: 11
      customerID  gender  SeniorCitizen Partner Dependents  tenure  \
488   4472-LVYGI  Female              0     Yes        Yes       0   
753   3115-CZMZD    Male              0      No        Yes       0   
936   5709-LVOEQ  Female              0     Yes        Yes       0   
1082  4367-NUYAO    Male              0     Yes        Yes       0   
1340  1371-DWPAZ  Female              0     Yes        Yes       0   
3331  7644-OMVMY    Male              0     Yes        Yes       0   
3826  3213-VVOLG    Male              0     Yes        Yes       0   
4380  2520-SGTTA  Female              0     Yes        Yes       0   
5218  2923-ARZLG    Male              0     Yes        Yes       0   
6670  4075-WKNIU  Female              0     Yes        Yes       0   
6754  2775-SEFEE    Male              0      No        Yes       0   

     PhoneService     MultipleLines InternetService       OnlineSecurity  ...  \
488            No  No phone service        

**Discovery**: 11 rows had missing TotalCharges values. Upon investigation, all 11 rows had tenure = 0, meaning these were brand-new customers who hadn't been billed yet.

**Decision**: Since customers with zero tenure haven't had time to churn or remain loyal, they don't provide meaningful insights for churn prediction. I removed these rows.


In [27]:
df = df.drop(df[mv].index)
df["TotalCharges"] = df["TotalCharges"].astype('float')
print("TotalCharges successfully converted to float")

TotalCharges successfully converted to float


**Additional Data Quality Checks**
I verified that no other numeric columns contained unexpected negative values:

In [28]:
print((df["tenure"] < 0).sum())
print((df["MonthlyCharges"] < 0).sum())
print((df["TotalCharges"] < 0).sum())
print((df["SeniorCitizen"] < 0).sum())

0
0
0
0


I then analyzed the rest of the data set, focusing on the integer and float specific columns to ensure there's no negative values in columns where they wouldn't make sense.

**Result**: No negative values found. All numeric data is valid.
**Removing Non-Predictive Features**
I removed the customerID column as it serves only as a record identifier and has no predictive value for churn modeling.

In [29]:
df = df.drop(columns=["customerID"])
df.to_csv('/Users/arkyaghosh/churn_project_data/processed/telco_clean.csv', index=False)

## 4. Final Cleaned Data

In [30]:
df_cleaned = pd.read_csv('/Users/arkyaghosh/churn_project_data/processed/telco_clean.csv')
print(f"Final dataset shape: {df_cleaned.shape}")
print(df_cleaned.head())

Final dataset shape: (7032, 20)
   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   
3  No phone service             DSL            Yes           No   
4                No     Fiber optic             No           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   


### Summary of Cleaning Steps

| Issue | Action | Impact |
|-------|--------|--------|
| 11 rows with tenure=0 | Removed (incomplete churn data) | Reduced from 7,043 to 7,032 rows |
| TotalCharges as string | Converted to float | Now usable for numeric analysis |
| customerID | Removed | Non-predictive identifier |
| Negative values | Verified none exist | Data integrity confirmed |

**Conclusion**: The dataset is now clean, consistent, and ready for exploratory data analysis.